# 03 – VeritasCarbon VLDB Experiments

This notebook runs all experiments required for the VLDB 2026 submission:

| Step | Module | Description |
|------|--------|-------------|
| 1 | `baseline_local_03_01.py` | 3 baselines (Direct / Self-Instruct / WizardLM-Evol) using local Qwen2-72B |
| 2 | `intrinsic_evaluation_03_03.py` | Intrinsic quality metrics (ROUGE-L, BLEU-4, Distinct-n, domain relevance, etc.) |
| 3 | `ablation_local_03_02.py` | 4 ablation dimensions (expert count / collaboration / feedback / knowledge) |
| 4 | `dataset_statistics_03_04.py` | Corpus & QA statistics tables + paper-ready figures |

**All experiments use the same local Qwen2-72B-Instruct (4-bit quantized) for fair comparison.**

> **Naming note**: Our framework is named **CoDE** (Council of Domain Experts), distinct from Chain-of-Experts (Wang et al., 2024) which addresses MoE model architecture.

---

## 0. Environment Setup

In [1]:
# ============================================================================
# 0. Environment check & path config (HPC: /hpc2hdd/home/yjiang909)
# ============================================================================

import os, sys, json, time, logging, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(name)s %(levelname)s %(message)s')

# ---- Project root ----
PROJECT_ROOT = Path('/hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB')
SRC_PATH     = PROJECT_ROOT / 'src'
DATA_PATH    = PROJECT_ROOT / 'data'
RESULTS_PATH = PROJECT_ROOT / 'results'

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
    print(f'Added {SRC_PATH} to sys.path')

# ---- Model path (HPC) ----
MODEL_PATH = '/hpc2hdd/home/yjiang909/models/Qwen2-72B-Instruct/Qwen/Qwen2-72B-Instruct'

print('='*80)
print('Environment')
print('='*80)
print(f'  Python   : {sys.executable}')
print(f'  Project  : {PROJECT_ROOT}')
print(f'  Model    : {MODEL_PATH}')
print(f'  Model exists: {Path(MODEL_PATH).exists()}')

import torch
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        total = torch.cuda.get_device_properties(i).total_memory / 1024**3
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}  ({total:.0f} GB)')
else:
    print('  GPU: not available (CPU mode – will be very slow)')
print('='*80)

Added /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/src to sys.path
Environment
  Python   : /hpc2hdd/home/yjiang909/VeritasCarbon_venv/bin/python
  Project  : /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB
  Model    : /hpc2hdd/home/yjiang909/models/Qwen2-72B-Instruct/Qwen/Qwen2-72B-Instruct
  Model exists: True
  GPU 0: NVIDIA A800-SXM4-80GB  (79 GB)


## 1. Load Qwen2-72B-Instruct (Unsloth / Standard)

In [2]:
# ============================================================================
# 1. Load model  (same logic as notebook 02)
# ============================================================================
import torch, os

USE_UNSLOTH = True
try:
    from unsloth import FastLanguageModel
    USE_UNSLOTH = True
    print('Unsloth detected – using acceleration')
except ImportError:
    print('Unsloth not available – standard transformers mode')

if USE_UNSLOTH:
    os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
    if not os.environ.get('HF_ENDPOINT'):
        os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True,
        device_map={'': 0},
        local_files_only=False,
    )
    FastLanguageModel.for_inference(model)
    print('Model loaded (Unsloth 4-bit)')
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=True,
    )
    print('Model loaded (standard bfloat16)')

# Quick sanity check
try:
    params = model.num_parameters() / 1e9
except Exception:
    params = 72.0
print(f'Params: {params:.1f}B   dtype: {model.dtype}')
if torch.cuda.is_available():
    alloc = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM: {alloc:.1f} / {total:.0f} GB')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth detected – using acceleration
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.1.2: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A800-SXM4-80GB. Num GPUs = 1. Max memory: 79.252 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 37/37 [08:17<00:00, 13.44s/it]


Model loaded (Unsloth 4-bit)
Params: 72.7B   dtype: torch.bfloat16
VRAM: 38.4 / 79 GB


## 2. Initialize Expert System Components

In [3]:
# ============================================================================
# 2. Expert system – reuse LocalQwenExpertAdapter from notebook 02
# ============================================================================

from instruction_generation.expert_agents_02_04 import BaseExpert
from instruction_generation.coe_framework_02_03 import COEFramework, ExpertOutput
from instruction_generation.expert_selector_02_01 import ExpertSelector
from instruction_generation.domain_knowledge_02_02 import DomainKnowledgeInjector

# ---------- LocalQwenExpertAdapter ----------
class LocalQwenExpertAdapter(BaseExpert):
    """Adapter: bridge local Qwen2-72B to the BaseExpert interface (no API)."""

    def __init__(self, name, local_model, local_tokenizer,
                 temperature=0.7, max_tokens=1024):
        self.name = name
        self.local_model = local_model
        self.local_tokenizer = local_tokenizer
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.api_provider = 'local_qwen'
        self.model_name = 'Qwen2-72B-Instruct'

    def _call_local_model(self, prompt, max_tokens=None):
        messages = [{'role': 'user', 'content': prompt}]
        text = self.local_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        inputs = self.local_tokenizer([text], return_tensors='pt').to(self.local_model.device)
        with torch.no_grad():
            gen = self.local_model.generate(
                **inputs,
                max_new_tokens=max_tokens or self.max_tokens,
                temperature=self.temperature, top_p=0.9, do_sample=True)
        gen_only = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
        return self.local_tokenizer.batch_decode(gen_only, skip_special_tokens=True)[0].strip()

    def get_prompt_template(self, chunk_text, context=None, dynamic_instruction=None):
        di = dynamic_instruction or (context.get('dynamic_instruction', '') if context else '')
        kn = context.get('domain_knowledge', '') if context else ''
        return (f'You are an ESG expert. Generate a high-quality Q&A pair.\n\n'
                f'{di}\n\n{kn}\n\nText:\n{chunk_text}\n\n'
                f'Output format:\nInstruction: [question]\nResponse: [answer]')

    def parse_response(self, response):
        instruction, answer = '', ''
        for line in response.split('\n'):
            if line.lower().startswith('instruction:') or line.startswith('指令'):
                instruction = line.split(':', 1)[-1].strip()
            elif line.lower().startswith('response:') or line.lower().startswith('answer:') or line.startswith('回答'):
                answer = line.split(':', 1)[-1].strip()
        if not instruction or len(instruction) < 5:
            instruction = 'Analyze the following ESG-related content'
        if not answer or len(answer) < 10:
            answer = response.strip()
        return ExpertOutput(expert_name=self.name, instruction=instruction,
                            response=answer, quality_score=0.0,
                            metadata={'expert_type': 'local_qwen'})

    def generate(self, chunk_text, context=None):
        prompt = self.get_prompt_template(chunk_text, context)
        resp = self._call_local_model(prompt)
        return self.parse_response(resp)

# ---------- 11 per-expert subclasses ----------
class LocalQAExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('QA Expert', m, t)
class LocalSummaryExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Summary Expert', m, t)
class LocalExtractionExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Extraction Expert', m, t)
class LocalClassificationExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Classification Expert', m, t)
class LocalAnalysisExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Analysis Expert', m, t)
class LocalTemporalAnalysisExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Temporal Analysis Expert', m, t)
class LocalBenchmarkComparisonExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Benchmark Comparison Expert', m, t)
class LocalGreenwashingDetectionExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Greenwashing Detection Expert', m, t)
class LocalStandardAlignmentExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Standard Alignment Expert', m, t)
class LocalConsistencyVerificationExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Consistency Verification Expert', m, t)
class LocalKnowledgeGraphExpert(LocalQwenExpertAdapter):
    def __init__(self, m, t): super().__init__('Knowledge Graph Expert', m, t)

# ---------- Instantiate all 11 experts ----------
experts = {
    'qa_expert':                       LocalQAExpert(model, tokenizer),
    'summary_expert':                  LocalSummaryExpert(model, tokenizer),
    'extraction_expert':               LocalExtractionExpert(model, tokenizer),
    'classification_expert':           LocalClassificationExpert(model, tokenizer),
    'analysis_expert':                 LocalAnalysisExpert(model, tokenizer),
    'temporal_analysis_expert':        LocalTemporalAnalysisExpert(model, tokenizer),
    'benchmark_comparison_expert':     LocalBenchmarkComparisonExpert(model, tokenizer),
    'greenwashing_detection_expert':   LocalGreenwashingDetectionExpert(model, tokenizer),
    'standard_alignment_expert':       LocalStandardAlignmentExpert(model, tokenizer),
    'consistency_verification_expert': LocalConsistencyVerificationExpert(model, tokenizer),
    'knowledge_graph_expert':          LocalKnowledgeGraphExpert(model, tokenizer),
}
print(f'Created {len(experts)} local expert instances')

# ---------- System components ----------
expert_selector = ExpertSelector(
    min_experts=1, max_experts=3,
    use_layered_selection=True,
    layered_config={
        'base_layer_required': True,
        'analysis_layer_threshold': 0.6,
        'verification_layer_threshold': 0.7,
        'graph_layer_threshold': 0.8,
    }
)

knowledge_base_path = PROJECT_ROOT / 'data' / 'knowledge_base'
knowledge_injector = DomainKnowledgeInjector(
    knowledge_base_path=str(knowledge_base_path) if knowledge_base_path.exists() else None,
    max_knowledge_items=5
)

coe_framework = COEFramework(
    experts=experts,
    enable_collaboration=True,
    collaboration_mode='sequential',
    max_iterations=2,
    quality_threshold=0.7
)
print('ExpertSelector / DomainKnowledgeInjector / COEFramework initialized')

2026-02-25 13:10:53,308 instruction_generation.expert_selector_02_01 INFO Loading classifier model: bert-base-chinese
2026-02-25 13:10:53,309 instruction_generation.expert_selector_02_01 WARNING Classifier not trained; using rule-based selection
2026-02-25 13:10:53,312 instruction_generation.domain_knowledge_02_02 INFO Loaded knowledge base: 0 entities, 0 relations


Created 11 local expert instances
ExpertSelector / DomainKnowledgeInjector / COEFramework initialized


## 3. Load Chunks for Experiments

We sample a fixed set of chunks for reproducibility.  
- **Baselines**: 500 chunks  
- **Ablation**: 200 chunks (subset of baseline set)

In [4]:
# ============================================================================
# 3. Load chunks
# ============================================================================
import json, random

CHUNKS_FILE = str(DATA_PATH / 'processed_corpus' / 'chunks_clean_fixed.jsonl')

# Try smaller sampled file first
sampled_file = DATA_PATH / 'processed_corpus' / 'chunks_sampled_20000_by_year.jsonl'
if sampled_file.exists():
    CHUNKS_FILE = str(sampled_file)
    print(f'Using sampled file: {sampled_file.name}')

print(f'Loading chunks from: {CHUNKS_FILE}')
all_chunks = []
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            all_chunks.append(json.loads(line))
print(f'Total chunks loaded: {len(all_chunks):,}')

# Reproducible sampling
random.seed(42)
BASELINE_N = 500
ABLATION_N = 200

baseline_chunks = random.sample(all_chunks, min(BASELINE_N, len(all_chunks)))
ablation_chunks = baseline_chunks[:ABLATION_N]   # subset for speed

print(f'Baseline sample: {len(baseline_chunks)} chunks')
print(f'Ablation sample: {len(ablation_chunks)} chunks')

Using sampled file: chunks_sampled_20000_by_year.jsonl
Loading chunks from: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/data/processed_corpus/chunks_sampled_20000_by_year.jsonl
Total chunks loaded: 20,000
Baseline sample: 500 chunks
Ablation sample: 200 chunks


## 4. Run Baselines (baseline_local_03_01)

Three baselines all using local Qwen2-72B-Instruct for fair comparison:  
- **B1 – Direct Prompting**: single-turn zero-shot  
- **B2 – Self-Instruct**: seed templates + random selection  
- **B3 – WizardLM-Evol**: iterative multi-step refinement

In [5]:

# ============================================================================
# 4. Baselines
# ============================================================================
import json, shutil
from pathlib import Path
from instruction_generation.baseline_local_03_01 import LocalBaselineExperiment

# LocalBaselineExperiment(model, tokenizer) – not llm= keyword
baseline_exp = LocalBaselineExperiment(model, tokenizer)

baselines_dir = RESULTS_PATH / 'baselines'
baselines_dir.mkdir(parents=True, exist_ok=True)

print('Running 3 baselines …')
print()

stats = baseline_exp.run_all_baselines(
    baseline_chunks,
    output_dir=str(baselines_dir),
    checkpoint_every=50,
)

# Load results back (saved as {method}_results.jsonl by run_all_baselines)
def _load_jsonl(path):
    records = []
    p = Path(path)
    if p.exists():
        with open(p, encoding='utf-8') as fh:
            for line in fh:
                if line.strip():
                    records.append(json.loads(line))
    return records

b1 = _load_jsonl(baselines_dir / 'direct_prompting_results.jsonl')
b2 = _load_jsonl(baselines_dir / 'self_instruct_results.jsonl')
b3 = _load_jsonl(baselines_dir / 'wizardlm_evol_results.jsonl')

# Create canonical names expected by the evaluation cell
for src, dst in [
    ('direct_prompting_results.jsonl', 'direct_prompting.jsonl'),
    ('self_instruct_results.jsonl',    'self_instruct.jsonl'),
    ('wizardlm_evol_results.jsonl',    'wizardlm_evol.jsonl'),
]:
    src_p, dst_p = baselines_dir / src, baselines_dir / dst
    if src_p.exists():
        shutil.copy2(src_p, dst_p)

print(f'[B1] Direct Prompting:  {len(b1)} records')
print(f'[B2] Self-Instruct:     {len(b2)} records')
print(f'[B3] WizardLM-Evol:     {len(b3)} records')
print()
print(f'Baseline outputs saved to: {baselines_dir}')


2026-02-25 13:10:54,123 instruction_generation.baseline_local_03_01 INFO [direct_prompting] Resuming: found 500 existing results, 500 chunk IDs already processed
2026-02-25 13:10:54,125 instruction_generation.baseline_local_03_01 INFO Running baseline: direct_prompting — 0 remaining / 500 total chunks
2026-02-25 13:10:54,126 instruction_generation.baseline_local_03_01 INFO [direct_prompting] All chunks already processed, skipping.


Running 3 baselines …



2026-02-25 13:10:54,224 instruction_generation.baseline_local_03_01 INFO [self_instruct] Resuming: found 500 existing results, 500 chunk IDs already processed
2026-02-25 13:10:54,225 instruction_generation.baseline_local_03_01 INFO Running baseline: self_instruct — 0 remaining / 500 total chunks
2026-02-25 13:10:54,226 instruction_generation.baseline_local_03_01 INFO [self_instruct] All chunks already processed, skipping.
2026-02-25 13:10:54,341 instruction_generation.baseline_local_03_01 INFO [wizardlm_evol] Resuming: found 500 existing results, 500 chunk IDs already processed
2026-02-25 13:10:54,342 instruction_generation.baseline_local_03_01 INFO Running baseline: wizardlm_evol — 0 remaining / 500 total chunks
2026-02-25 13:10:54,343 instruction_generation.baseline_local_03_01 INFO [wizardlm_evol] All chunks already processed, skipping.
2026-02-25 13:10:54,345 instruction_generation.baseline_local_03_01 INFO Summary saved: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/results/b

[B1] Direct Prompting:  500 records
[B2] Self-Instruct:     500 records
[B3] WizardLM-Evol:     500 records

Baseline outputs saved to: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/results/baselines


## 5. Intrinsic Quality Evaluation (intrinsic_evaluation_03_03)

Metrics computed on CoDE + each baseline:  
ROUGE-L · BLEU-4 · Distinct-2 · Domain Relevance · Factcheck Overlap · Structural Completeness

In [7]:
# ============================================================================
# 5. Intrinsic evaluation
# ============================================================================
from instruction_generation.intrinsic_evaluation_03_03 import (
    IntrinsicEvaluator, run_comparison
)

# Paths to CoDE QA files (generated by notebook 02)
coe_files = [
    str(DATA_PATH / 'instructions' / 'qa_pairs_complete_v3_1.5w.jsonl'),
    str(DATA_PATH / 'instructions' / 'qa_pairs_complete_v3_2w.jsonl'),
]

# Paths to baseline & ablation directories
baselines_dir = RESULTS_PATH / 'baselines'
ablation_dir  = RESULTS_PATH / 'ablation'
output_path   = str(RESULTS_PATH / 'outputs' / 'intrinsic_comparison.json')

print('Running intrinsic evaluation …')
comparison = run_comparison(
    coe_files=coe_files,
    baseline_dir=str(baselines_dir),
    ablation_dir=str(ablation_dir),
    output_path=output_path,
    max_per_dataset=2000,
)

# Pretty-print
import pandas as pd
df = pd.DataFrame(comparison).T
print()
print(df.to_string())

Running intrinsic evaluation …


2026-02-25 13:14:19,172 absl INFO Using default tokenizer.
2026-02-25 13:14:19,175 instruction_generation.intrinsic_evaluation_03_03 INFO ROUGE scorer loaded
2026-02-25 13:14:24,095 instruction_generation.intrinsic_evaluation_03_03 INFO CoDE: 2000 records evaluated
2026-02-25 13:14:25,461 instruction_generation.intrinsic_evaluation_03_03 INFO Baseline direct_prompting: 500 records evaluated
2026-02-25 13:14:26,471 instruction_generation.intrinsic_evaluation_03_03 INFO Baseline self_instruct: 500 records evaluated
2026-02-25 13:14:30,430 instruction_generation.intrinsic_evaluation_03_03 INFO Baseline wizardlm_evol: 500 records evaluated
2026-02-25 13:14:30,434 instruction_generation.intrinsic_evaluation_03_03 INFO Comparison saved: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/results/outputs/intrinsic_comparison.json
2026-02-25 13:14:30,436 instruction_generation.intrinsic_evaluation_03_03 INFO CSV saved: /hpc2hdd/home/yjiang909/Veritas/VeritasCarbon_VLDB/results/outputs/intrinsic


                             label     n avg_instruction_length avg_response_length rouge_l   bleu4 distinct_1 distinct_2 distinct_3 domain_relevance factcheck structural_completeness avg_quality_score
CoDE (ours)            CoDE (ours)  2000                  106.7               377.7  0.3433  0.1936     0.0032     0.1065     0.3054           0.3614    0.9463                   0.998            0.6664
direct_prompting  direct_prompting   500                  128.1               959.6  0.0892  0.0456     0.0026     0.0221     0.0501           0.0216    0.8692                     1.0              None
self_instruct        self_instruct   500                   92.8               706.1  0.0401  0.0096     0.0014      0.007     0.0264            0.004    0.9362                     1.0              None
wizardlm_evol        wizardlm_evol   500                 2581.6              3201.9  0.0336  0.0118     0.0008     0.0069      0.018           0.0228    0.5682                     1.0        

## 6. Ablation Studies (ablation_local_03_02)

Four ablation dimensions, each varying one factor:  
1. **Expert count**: 1 / 2 / 3 / 5 experts per chunk  
2. **Collaboration mode**: none / sequential / parallel  
3. **Feedback rounds**: 0 / 1 / 2  
4. **Knowledge injection**: on / off

In [ ]:
# ============================================================================
# 6. Ablation studies
# ============================================================================
from instruction_generation.ablation_local_03_02 import LocalAblationStudy
from instruction_generation.coe_framework_02_03 import COEFramework, ExpertOutput

ablation = LocalAblationStudy(
    model=model,
    tokenizer=tokenizer,
    experts=experts,
    expert_selector=expert_selector,
    knowledge_injector=knowledge_injector,
    coe_framework_cls=COEFramework,
    expert_output_cls=ExpertOutput,
)

ablation_output_dir = str(RESULTS_PATH / 'ablation')
print('Running ablation studies on', len(ablation_chunks), 'chunks …')
print()

# A1 – Expert count
print('[A1] Expert count ablation')
a1 = ablation.run_expert_count_ablation(ablation_chunks, output_dir=f'{ablation_output_dir}/expert_count')
print(f'  Conditions: {list(a1.keys())}')

# A2 – Collaboration mode
print('[A2] Collaboration mode ablation')
a2 = ablation.run_collaboration_ablation(ablation_chunks, output_dir=f'{ablation_output_dir}/collaboration')
print(f'  Conditions: {list(a2.keys())}')

# A3 – Feedback rounds
print('[A3] Feedback rounds ablation')
a3 = ablation.run_feedback_ablation(ablation_chunks, output_dir=f'{ablation_output_dir}/feedback')
print(f'  Conditions: {list(a3.keys())}')

# A4 – Knowledge injection
print('[A4] Knowledge injection ablation')
a4 = ablation.run_knowledge_ablation(ablation_chunks, output_dir=f'{ablation_output_dir}/knowledge')
print(f'  Conditions: {list(a4.keys())}')

print()
print(f'Ablation outputs saved to: {ablation_output_dir}')

TypeError: LocalAblationStudy.__init__() got an unexpected keyword argument 'output_dir'

## 7. Dataset Statistics & Paper Figures (dataset_statistics_03_04)

In [ ]:
# ============================================================================
# 7. Dataset statistics + figures
# ============================================================================
from instruction_generation.dataset_statistics_03_04 import (
    generate_all_tables_and_figures
)

FIGURES_DIR = str(RESULTS_PATH / 'figures_and_tables')

generate_all_tables_and_figures(
    raw_corpus_dir=str(DATA_PATH / 'raw_corpus'),
    chunks_file=CHUNKS_FILE,
    qa_files=coe_files,
    comparison_json=str(RESULTS_PATH / 'outputs' / 'intrinsic_comparison.json'),
    ablation_summary=str(RESULTS_PATH / 'ablation' / 'ablation_summary.json'),
    output_dir=FIGURES_DIR,
)

print()
print(f'All tables and figures saved to: {FIGURES_DIR}')

## 8. Display Results

### 8.1 Corpus Statistics (Table 1)

In [ ]:
# ============================================================================
# 8.1  Table 1: Corpus statistics
# ============================================================================
import pandas as pd, json

table1_path = RESULTS_PATH / 'figures_and_tables' / 'table1_corpus_statistics.csv'
if table1_path.exists():
    df1 = pd.read_csv(table1_path)
    display(df1)
else:
    t1_json = RESULTS_PATH / 'figures_and_tables' / 'table1_corpus_statistics.json'
    if t1_json.exists():
        with open(t1_json) as f:
            print(json.dumps(json.load(f), indent=2, ensure_ascii=False))
    else:
        print('Table 1 not generated yet – run cell 7 first')

### 8.2 QA Dataset Overview (Table 2)

In [ ]:
# ============================================================================
# 8.2  Table 2: QA overview
# ============================================================================
table2_path = RESULTS_PATH / 'figures_and_tables' / 'table2_qa_overview.csv'
if table2_path.exists():
    df2 = pd.read_csv(table2_path)
    display(df2)
else:
    print('Table 2 not generated yet – run cell 7 first')

### 8.3 Expert Distribution (Table 2 detail)

In [ ]:
# ============================================================================
# 8.3  Expert distribution table
# ============================================================================
ed_path = RESULTS_PATH / 'figures_and_tables' / 'table2_expert_distribution.csv'
if ed_path.exists():
    df_ed = pd.read_csv(ed_path)
    display(df_ed)
else:
    print('Expert distribution not generated yet')

### 8.4 Intrinsic Evaluation Comparison (CoDE vs Baselines)

In [ ]:
# ============================================================================
# 8.4  Intrinsic comparison table
# ============================================================================
comp_csv = RESULTS_PATH / 'outputs' / 'intrinsic_comparison.csv'
if comp_csv.exists():
    df_comp = pd.read_csv(comp_csv)
    display(df_comp)
else:
    comp_json = RESULTS_PATH / 'outputs' / 'intrinsic_comparison.json'
    if comp_json.exists():
        with open(comp_json) as f:
            cmp = json.load(f)
        display(pd.DataFrame(cmp).T)
    else:
        print('Comparison not generated yet – run cell 5 first')

### 8.5 Paper Figures

In [ ]:
# ============================================================================
# 8.5  Display all generated figures (PNG previews)
# ============================================================================
from pathlib import Path
from IPython.display import Image, display as ipy_display

fig_dir = RESULTS_PATH / 'figures_and_tables'
png_files = sorted(fig_dir.glob('fig*.png')) if fig_dir.exists() else []

if png_files:
    for png in png_files:
        print(f'\n--- {png.name} ---')
        ipy_display(Image(filename=str(png), width=700))
else:
    print('No figures generated yet – run cell 7 first')

## 9. Summary

After all cells have been executed, the following artefacts are available:

| Artefact | Path |
|----------|------|
| Baseline QA outputs | `results/baselines/*.jsonl` |
| Ablation QA outputs | `results/ablation/*.jsonl` |
| Intrinsic comparison (JSON + CSV) | `results/outputs/intrinsic_comparison.*` |
| Corpus statistics (Table 1) | `results/figures_and_tables/table1_*` |
| QA statistics (Table 2) | `results/figures_and_tables/table2_*` |
| Figures 1-6 (PNG + PDF) | `results/figures_and_tables/fig*` |

Copy figures and tables to `paper/figures/` for LaTeX inclusion.

In [ ]:
# Copy figures to paper directory
import shutil

paper_figs = PROJECT_ROOT / 'paper' / 'figures'
paper_figs.mkdir(parents=True, exist_ok=True)

fig_dir = RESULTS_PATH / 'figures_and_tables'
if fig_dir.exists():
    for f in fig_dir.iterdir():
        if f.suffix in ('.png', '.pdf', '.csv'):
            shutil.copy2(f, paper_figs / f.name)
    print(f'Copied {len(list(paper_figs.iterdir()))} files to {paper_figs}')
else:
    print('No figures to copy – run experiment cells first')

print()
print('='*80)
print('VLDB experiment pipeline complete')
print('='*80)